# 03 - Regression Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../Data/creditcard_clean.csv")
v_cols = [f"V{i}" for i in range(1, 29)]
X = df[v_cols]
y = df["Amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 1. Correlation with Amount

In [ ]:
corr_amt = df[v_cols + ["Amount"]].corr()["Amount"].drop("Amount")
corr_amt = corr_amt.reindex(corr_amt.abs().sort_values(ascending=False).index)
print(corr_amt.head(8).round(3))

plt.figure(figsize=(8, 7))
corr_amt.sort_values().plot(kind="barh")
plt.title("Correlation of V-features with Amount")
plt.show()

best_feat = corr_amt.index[0]
top5 = corr_amt.index[:5].tolist()
print("Best single feature:", best_feat, "| Top 5:", top5)

In [ ]:
def reg_metrics(name, model, Xtr, Xte):
    pred_tr = model.predict(Xtr)
    pred_te = model.predict(Xte)
    mse = mean_squared_error(y_test, pred_te)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred_te),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2_test": r2_score(y_test, pred_te),
        "R2_train": r2_score(y_train, pred_tr),
    }

results = []

## 2. Simple Linear Regression

In [ ]:
slr = LinearRegression().fit(X_train[[best_feat]], y_train)
print(f"Amount = {slr.intercept_:.3f} + ({slr.coef_[0]:.3f}) * {best_feat}")
results.append(reg_metrics(f"Simple Linear ({best_feat})", slr, X_train[[best_feat]], X_test[[best_feat]]))

plt.figure(figsize=(7, 5))
sample = X_test[[best_feat]].join(y_test).sample(3000, random_state=1)
plt.scatter(sample[best_feat], sample["Amount"], s=8, alpha=0.4)
xs = np.linspace(sample[best_feat].min(), sample[best_feat].max(), 100)
plt.plot(xs, slr.predict(pd.DataFrame({best_feat: xs})), color="red")
plt.xlabel(best_feat); plt.ylabel("Amount"); plt.title("Simple Linear Regression")
plt.show()

## 3. Multiple Linear Regression

In [ ]:
mlr = LinearRegression().fit(X_train, y_train)
results.append(reg_metrics("Multiple Linear (V1-V28)", mlr, X_train, X_test))

coef = pd.Series(mlr.coef_, index=v_cols).sort_values(key=abs, ascending=False)
print("Intercept:", round(mlr.intercept_, 3))
coef.head(10).round(3)

## 4. Polynomial Regression

In [ ]:
X5_train, X5_test = X_train[top5], X_test[top5]

lin5 = LinearRegression().fit(X5_train, y_train)
results.append(reg_metrics("Linear (top 5)", lin5, X5_train, X5_test))

for deg in [2, 3]:
    poly = make_pipeline(PolynomialFeatures(degree=deg, include_bias=False), LinearRegression())
    poly.fit(X5_train, y_train)
    results.append(reg_metrics(f"Poly deg {deg} (top 5)", poly, X5_train, X5_test))

## 5. Metrics Comparison

In [ ]:
res = pd.DataFrame(results).set_index("Model").round(4)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
res["RMSE"].plot(kind="barh", ax=ax[0]); ax[0].set_title("RMSE")
res["R2_test"].plot(kind="barh", ax=ax[1], color="seagreen"); ax[1].set_title("R2 test")
plt.tight_layout(); plt.show()

## 6. Residual Plot

In [ ]:
pred = mlr.predict(X_test)
resid = y_test - pred

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
idx = np.random.RandomState(0).choice(len(y_test), 4000, replace=False)
ax[0].scatter(np.asarray(y_test)[idx], pred[idx], s=6, alpha=0.4)
ax[0].plot([0, y_test.max()], [0, y_test.max()], "r--")
ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted"); ax[0].set_title("Actual vs Predicted")
ax[1].scatter(pred[idx], np.asarray(resid)[idx], s=6, alpha=0.4)
ax[1].axhline(0, color="r", ls="--")
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Residual"); ax[1].set_title("Residuals vs Predicted")
ax[2].hist(resid, bins=80); ax[2].set_title("Residual Distribution")
plt.tight_layout(); plt.show()